#### Task 5: Correlation between Tenure and Churn (Churn converted to 1 = Yes, 0 = No).

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
CANON = ["CustomerID", "Gender", "SeniorCitizen", "Partner", "Dependents", "Tenure", "PhoneService",
         "InternetService", "Contract", "MonthlyCharges", "TotalCharges", "Churn"]
 
 
def load_data():
    """Load the real Telco-Customer-Churn.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("Telco-Customer-Churn.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv", "Telco-Customer-Churn_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {name}: {len(df):,} rows" + ("  (SIMULATED STAND-IN DATA, not the real file)" if "sample" in name else ""))
            break
    else:
        raise FileNotFoundError("Put Telco-Customer-Churn.csv (or run generate_sample_data.py) in the same folder.")
    df = df.rename(columns={c: next((k for k in CANON if k.lower() == c.lower()), c) for c in df.columns})   # customerID -> CustomerID, tenure -> Tenure
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")        # blank strings become NaN
    return df
 
from scipy import stats
 
df = load_data()
 
# Excel equivalent: new column =IF(Churn="Yes",1,0) then =CORREL(Tenure_range, ChurnNum_range)
df["Churn_Num"] = np.where(df["Churn"] == "Yes", 1, 0)
r = df["Tenure"].corr(df["Churn_Num"])
print(f"\nCORREL(Tenure, Churn_Num) = {r:.4f}")
print(f"R-squared = {r ** 2:.3f}")
rp, p = stats.pointbiserialr(df["Churn_Num"], df["Tenure"])
print(f"scipy point-biserial check: r = {rp:.4f}, p-value = {p:.4g}")
 
print(f"\nAverage tenure of churned customers     = {df.loc[df['Churn'] == 'Yes', 'Tenure'].mean():.1f} months")
print(f"Average tenure of non-churned customers = {df.loc[df['Churn'] == 'No', 'Tenure'].mean():.1f} months")
 
strength = "strong" if abs(r) >= 0.6 else "moderate" if abs(r) >= 0.3 else "weak"
direction = "negative" if r < 0 else "positive"
print(f"\nInterpretation: a {strength} {direction} correlation. "
      + ("Customers with longer tenure are less likely to churn; churn is concentrated among newer customers."
         if r < 0 else "Longer tenure goes with higher churn."))
print("Note: correlation shows association, not cause (e.g. contract type also affects both tenure and churn).")

Loaded Telco-Customer-Churn_sample.csv: 2,000 rows  (SIMULATED STAND-IN DATA, not the real file)

CORREL(Tenure, Churn_Num) = -0.3310
R-squared = 0.110
scipy point-biserial check: r = -0.3310, p-value = 2.489e-52

Average tenure of churned customers     = 16.5 months
Average tenure of non-churned customers = 32.3 months

Interpretation: a moderate negative correlation. Customers with longer tenure are less likely to churn; churn is concentrated among newer customers.
Note: correlation shows association, not cause (e.g. contract type also affects both tenure and churn).
